# Hackathon AI & ML — Advanced Model
**Stratégie** : LightGBM + CatBoost + Random Forest avec ensembling par averaging des probabilités.  
**Améliorations vs baseline** :
- Feature engineering : patterns de NaN, features démographiques clés
- Gestion du déséquilibre de classes explicite
- Threshold optimal sur OOF plutôt que 0.5
- Ensemble de 3 modèles diversifiés

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('data')
GROUP_ID = 'G1'
SUBMISSION_ID = '2'
RANDOM_STATE = 42
N_FOLDS = 5
MISSING_THRESHOLD = 0.70  # plus souple que le baseline à 0.80

print('Librairies chargées OK')

Librairies chargées OK


## 1. Chargement des données

In [2]:
data     = pd.read_csv(DATA_DIR / 'data.csv')
labels   = pd.read_csv(DATA_DIR / 'ground_truth_train.csv')
test_idx = pd.read_csv(DATA_DIR / 'test_indexes.csv', header=None, names=['SEQN'])
metadata = pd.read_csv(DATA_DIR / 'features_metadata.csv')

print(f'data         : {data.shape}')
print(f'ground_truth : {labels.shape}')
print(f'test_indexes : {test_idx.shape}')
print(f'metadata     : {metadata.shape}')

data         : (59064, 1540)
ground_truth : (54064, 2)
test_indexes : (5000, 1)
metadata     : (1539, 5)


## 2. Split train / test

In [3]:
test_seqn  = set(test_idx['SEQN'].values)
train_seqn = set(labels['SEQN'].values) - test_seqn

train_data = data[data['SEQN'].isin(train_seqn)].copy()
test_data  = data[data['SEQN'].isin(test_seqn)].copy()

train_df = train_data.merge(labels[labels['SEQN'].isin(train_seqn)], on='SEQN')

print(f'Train : {train_df.shape}  |  Test : {test_data.shape}')
print(f'Taux de mortalité : {train_df["MORTSTAT_2019"].mean():.2%}')

Train : (54064, 1541)  |  Test : (5000, 1540)
Taux de mortalité : 15.68%


## 3. Feature engineering

In [ ]:
# --- Colonnes par composante (via metadata) ---
meta_cols = metadata[['SAS', 'Component']].rename(columns={'SAS': 'col'})

labo_cols  = meta_cols[meta_cols['Component'] == 'Laboratory']['col'].tolist()
exam_cols  = meta_cols[meta_cols['Component'] == 'Examination']['col'].tolist()
quest_cols = meta_cols[meta_cols['Component'] == 'Questionnaire']['col'].tolist()

print(f'Labo: {len(labo_cols)} | Exam: {len(exam_cols)} | Quest: {len(quest_cols)}')

def add_missing_features(df, labo_cols, exam_cols, quest_cols):
    """Ajoute des features sur les patterns de valeurs manquantes."""
    all_feature_cols = [c for c in df.columns if c not in ['SEQN', 'MORTSTAT_2019']]
    df = df.copy()
    
    df['feat_nb_missing_total'] = df[all_feature_cols].isnull().sum(axis=1)
    df['feat_pct_missing_total'] = df['feat_nb_missing_total'] / len(all_feature_cols)
    
    labo_present  = [c for c in labo_cols  if c in df.columns]
    exam_present  = [c for c in exam_cols  if c in df.columns]
    quest_present = [c for c in quest_cols if c in df.columns]
    
    if labo_present:
        df['feat_nb_missing_labo']  = df[labo_present].isnull().sum(axis=1)
    if exam_present:
        df['feat_nb_missing_exam']  = df[exam_present].isnull().sum(axis=1)
    if quest_present:
        df['feat_nb_missing_quest'] = df[quest_present].isnull().sum(axis=1)
    
    return df

train_df  = add_missing_features(train_df,  labo_cols, exam_cols, quest_cols)
test_data = add_missing_features(test_data, labo_cols, exam_cols, quest_cols)

print('Features de NaN ajoutées')

## 4. Nettoyage : suppression colonnes trop vides

In [ ]:
missing_ratio = train_df.isnull().mean()
cols_to_drop  = missing_ratio[missing_ratio > MISSING_THRESHOLD].index.tolist()
cols_to_drop  = [c for c in cols_to_drop if c not in ['SEQN', 'MORTSTAT_2019']]

print(f'Colonnes supprimées (>{MISSING_THRESHOLD*100:.0f}% manquant) : {len(cols_to_drop)}')

train_clean = train_df.drop(columns=cols_to_drop)
test_clean  = test_data.drop(columns=[c for c in cols_to_drop if c in test_data.columns])

feature_cols    = [c for c in train_clean.columns if c not in ['SEQN', 'MORTSTAT_2019']]
X               = train_clean[feature_cols]
y               = train_clean['MORTSTAT_2019']
X_test          = test_clean[feature_cols]
test_seqn_order = test_clean['SEQN']

print(f'Features : {X.shape[1]}  |  Train : {X.shape[0]}  |  Test : {X_test.shape[0]}')
print(f'Class balance — 0: {(y==0).sum()}  1: {(y==1).sum()}  ratio: {y.mean():.3f}')

## 5. Entraînement — 5-fold CV avec OOF predictions

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Ratio pour scale_pos_weight
neg_pos_ratio = (y == 0).sum() / (y == 1).sum()
print(f'scale_pos_weight (neg/pos) : {neg_pos_ratio:.1f}')

# ---- Paramètres des modèles ----
lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'n_estimators': 2000,
    'learning_rate': 0.03,
    'num_leaves': 63,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'scale_pos_weight': neg_pos_ratio,
    'random_state': RANDOM_STATE,
    'verbose': -1,
    'n_jobs': -1,
}

cat_params = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 6,
    'loss_function': 'Logloss',
    'eval_metric': 'F1',
    'class_weights': [1, neg_pos_ratio],
    'random_seed': RANDOM_STATE,
    'verbose': 0,
    'early_stopping_rounds': 50,
}

rf_params = {
    'n_estimators': 500,
    'max_depth': 20,
    'min_samples_leaf': 10,
    'max_features': 'sqrt',
    'class_weight': 'balanced',
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
}

# ---- Stockage OOF et prédictions test ----
oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))
oof_rf  = np.zeros(len(X))

test_lgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))
test_rf  = np.zeros(len(X_test))

print('Début de la CV...')

In [ ]:
# ---- LightGBM ----
print('=== LightGBM ===')
f1_lgb = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(X_tr, y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(period=-1)])
    
    oof_lgb[val_idx] = m.predict_proba(X_val)[:, 1]
    test_lgb += m.predict_proba(X_test)[:, 1] / N_FOLDS
    
    f1 = f1_score(y_val, (oof_lgb[val_idx] > 0.5).astype(int))
    f1_lgb.append(f1)
    print(f'  Fold {fold+1} F1: {f1:.4f}')

print(f'  LightGBM CV F1: {np.mean(f1_lgb):.4f} ± {np.std(f1_lgb):.4f}')

In [ ]:
# ---- CatBoost ----
print('=== CatBoost ===')
f1_cat = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    m = CatBoostClassifier(**cat_params)
    m.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    
    oof_cat[val_idx] = m.predict_proba(X_val)[:, 1]
    test_cat += m.predict_proba(X_test)[:, 1] / N_FOLDS
    
    f1 = f1_score(y_val, (oof_cat[val_idx] > 0.5).astype(int))
    f1_cat.append(f1)
    print(f'  Fold {fold+1} F1: {f1:.4f}')

print(f'  CatBoost CV F1: {np.mean(f1_cat):.4f} ± {np.std(f1_cat):.4f}')

In [ ]:
# ---- Random Forest ----
print('=== Random Forest ===')
f1_rf = []

# RF ne gère pas les NaN — imputation par médiane
X_rf      = X.fillna(X.median())
X_test_rf = X_test.fillna(X.median())  # médiane du train

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_rf, y)):
    X_tr, X_val = X_rf.iloc[tr_idx], X_rf.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    m = RandomForestClassifier(**rf_params)
    m.fit(X_tr, y_tr)
    
    oof_rf[val_idx] = m.predict_proba(X_val)[:, 1]
    test_rf += m.predict_proba(X_test_rf)[:, 1] / N_FOLDS
    
    f1 = f1_score(y_val, (oof_rf[val_idx] > 0.5).astype(int))
    f1_rf.append(f1)
    print(f'  Fold {fold+1} F1: {f1:.4f}')

print(f'  Random Forest CV F1: {np.mean(f1_rf):.4f} ± {np.std(f1_rf):.4f}')

## 6. Ensembling + threshold optimal

In [ ]:
# --- Averaging des probabilités OOF ---
# Poids égaux — à ajuster selon les performances individuelles
w_lgb, w_cat, w_rf = 0.4, 0.4, 0.2

oof_ensemble  = w_lgb * oof_lgb  + w_cat * oof_cat  + w_rf * oof_rf
test_ensemble = w_lgb * test_lgb + w_cat * test_cat + w_rf * test_rf

# --- Recherche du threshold optimal sur OOF ---
thresholds = np.arange(0.20, 0.70, 0.01)
f1_by_threshold = [f1_score(y, (oof_ensemble > t).astype(int)) for t in thresholds]

best_threshold = thresholds[np.argmax(f1_by_threshold)]
best_f1_oof    = max(f1_by_threshold)

print(f'Threshold optimal : {best_threshold:.2f}')
print(f'F1 OOF ensemble   : {best_f1_oof:.4f}')
print(f'  (vs LGB seul    : {f1_score(y, (oof_lgb  > 0.5).astype(int)):.4f})')
print(f'  (vs CAT seul    : {f1_score(y, (oof_cat  > 0.5).astype(int)):.4f})')
print(f'  (vs RF seul     : {f1_score(y, (oof_rf   > 0.5).astype(int)):.4f})')

plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_by_threshold)
plt.axvline(best_threshold, color='red', linestyle='--', label=f'best={best_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('F1 OOF')
plt.title('F1 en fonction du seuil de décision (ensemble OOF)')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Soumission

In [ ]:
y_pred_test = (test_ensemble > best_threshold).astype(int)

submission = pd.DataFrame({
    'SEQN': test_seqn_order.values,
    'prediction': y_pred_test
}).sort_values('SEQN')

assert len(submission) == 5000, f'ERREUR : {len(submission)} lignes au lieu de 5000'

filename = f'{GROUP_ID}_{SUBMISSION_ID}.csv'
submission.to_csv(filename, index=False, header=False)

print(f'Fichier : {filename}')
print(f'Lignes  : {len(submission)}')
print(f'Répartition : {submission["prediction"].value_counts().to_dict()}')
print(submission.head())

## 8. Récapitulatif des performances

In [ ]:
print('=== Récapitulatif F1 OOF ===')
print(f'Baseline LightGBM (seuil 0.5) : 0.6945')
print(f'LightGBM avancé               : {np.mean(f1_lgb):.4f}')
print(f'CatBoost                      : {np.mean(f1_cat):.4f}')
print(f'Random Forest                 : {np.mean(f1_rf):.4f}')
print(f'Ensemble (seuil opt. {best_threshold:.2f})  : {best_f1_oof:.4f}')